In [0]:
%sql
CREATE CATALOG IF NOT EXISTS aircraft_data
MANAGED LOCATION 'abfss://projectdataset@project5kavan.dfs.core.windows.net/aircraft_data';

USE CATALOG aircraft_data;

CREATE SCHEMA IF NOT EXISTS dataset;

USE SCHEMA dataset;

CREATE VOLUME IF NOT EXISTS raw_data;

In [0]:
from pyspark.sql import functions as F
from datetime import datetime
import requests
import zipfile
import io
import os


base_path = "/Volumes/aircraft_data/dataset/raw_data/"

bronze_table = "aircraft_data.dataset.bronze_flight_data"

audit_table = "aircraft_data.dataset.pipeline_audit"

aircraft_table = "aircraft_data.dataset.airline_reference"

In [0]:
try:
    bronze_df = spark.table(bronze_table)
    bronze_exists = True
except Exception:
    bronze_df = None
    bronze_exists = False

In [0]:
try:
    audit_df = spark.table(audit_table)

    latest_month = (
        audit_df
        .filter(
            F.col("status") == "SUCCESS"
        )
        .select(
            F.max(
                F.concat(
                    F.col("file_year"),
                    F.lpad(
                        F.col("file_month"),
                        2,
                        "0"
                    )
                )
            ).alias("latest_month")
        )
        .collect()[0]["latest_month"]
    )

except Exception:
    latest_month = None


print("Latest processed month:", latest_month)

In [0]:
if latest_month:

    latest_year = int(latest_month[:4])
    latest_mon = int(latest_month[4:])

    if latest_mon == 12:
        start_year = latest_year + 1
        start_month = 1

    else:
        start_year = latest_year
        start_month = latest_mon + 1


else:

    # Initial Load
    start_year = 2015
    start_month = 1


print(
    "Starting ingestion from:",
    start_year,
    start_month
)

In [0]:
current_year = datetime.now().year
current_month = datetime.now().month

new_files = []

for year in range(start_year, current_year + 1):

    if year == start_year:
        month_start = start_month
    else:
        month_start = 1

    if year == current_year:
        month_end = current_month
    else:
        month_end = 12


    for month in range(month_start, month_end + 1):

        url = f"https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year}_{month}.zip"

        folder_path = f"{base_path}/{year}/{month:02d}"

        try:

            response = requests.get(url, timeout=60)
            response.raise_for_status()

            # create folder only after successful download
            os.makedirs(folder_path, exist_ok=True)

            with zipfile.ZipFile(io.BytesIO(response.content)) as z:

                csv_name = [
                    f for f in z.namelist()
                    if f.endswith(".csv")
                ][0]

                with z.open(csv_name) as infile:

                    file_path = f"{folder_path}/bts_{year}_{month:02d}.csv"

                    with open(file_path,"wb") as outfile:
                        outfile.write(infile.read())


            new_files.append(file_path)

            print(f"Downloaded {year}-{month:02d}")


        except Exception as e:

            print(
                f"No data available {year}-{month:02d}"
            )

In [0]:
if not new_files:
    dbutils.notebook.exit("No new files found")
else:
    new_df = (
    spark.read
    .option("header","true")
    .option("inferSchema","true")
    .option(
        "mode",
        "PERMISSIVE"
    )
    .csv(new_files)
)
    new_df = (
    new_df
    .withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "source_file",
        F.col("_metadata.file_path")
    )
    .withColumn(
        "folder_year",
        F.regexp_extract(
            F.col("source_file"),
            r"raw_data/(\d{4})/",
            1
        ).cast("int")
    )
    .withColumn(
        "folder_month",
        F.regexp_extract(
            F.col("source_file"),
            r"/(\d{2})/bts",
            1
        ).cast("int")
    )
)

In [0]:
if new_files:
    (
    new_df
    .dropDuplicates(
        [
            "source_file"
        ]
    )
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(
        bronze_table
    )
)
else:
    print("No new files to ingest. Bronze table is already up to date.")



In [0]:
display(
    spark.table(bronze_table)
    .select(
        "folder_year",
        "folder_month"
    )
    .distinct()
    .orderBy(
        "folder_year",
        "folder_month"
    )
)

In [0]:
#spark.table(bronze_table).count()

In [0]:
from pyspark.sql.functions import current_timestamp

if new_files:
    records_loaded = new_df.count()

    audit_df = (
        new_df
        .select(
            F.lit("Incremental_Bronze_Ingestion").alias("pipeline_name"),
            F.lit("BTS_On_Time_Performance").alias("source_name"),
            F.col("folder_year").cast("int").alias("file_year"),
            F.col("folder_month").cast("int").alias("file_month"),
            F.lit("SUCCESS").alias("status"),
            F.lit(records_loaded).alias("records_loaded"),
            current_timestamp().alias("ingestion_timestamp"),
            F.col("source_file")
        )
        .distinct()
    )
else:
    records_loaded = 0
    print("No new files. Skipping audit log.")

In [0]:
#audit pipeline
if new_files:
    (
        audit_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(
            audit_table
        )
    )

In [0]:
""""
if spark.catalog.tableExists("aircraft_data.dataset.pipeline_audit"):
    audit_df = spark.table(
        "aircraft_data.dataset.pipeline_audit"
    )

    last_processed = (
        audit_df
        .filter("status='SUCCESS'")
        .select(
            F.max(
                F.concat(
                    F.col("file_year"),
                    F.lpad(F.col("file_month"),2,"0")
                )
            )
        )
        .collect()[0][0]
    )

    print(last_processed)
else:
    print("Audit table does not exist yet. No records have been processed.") """

In [0]:
print("========== Bronze Pipeline Completed ==========")

print(
    f"""
Files processed : {len(new_files)}
Records loaded  : {records_loaded}
Completed time  : {datetime.now()}
"""
)

In [0]:


airline_reference = (spark.read
  .format("csv")
  .option("header", "true")
  .option("inferSchema", "true")
  .csv("/Volumes/aircraft_data/dataset/raw_data/"))
(
    airline_reference.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable('aircraft_table')
)